# 01 · Train the video transformer: EEG → CLIP, pretrained on THINGS-EEG2

**Question.** Can a transformer read our single-trial EEG if it first learns from a large public dataset? Trained only on our ~5,650 training recordings, a transformer stayed at chance whent ried. THINGS-EEG2 (Gifford et al., 2022) has ~660,000 single trials of visual responses, recorded at **the same 17 posterior channels in the same order** as ours (confirmed in Discord Mind Reading Channel at 17.10. 16:04). These channels lie over visual cortex, so THINGS-EEG2 can teach the visual part of our EEG.

**Model** (0.87 M parameters):

```
EEG: 17 channels × 0–0.8 s at 100 Hz
  → per-subject 17×17 channel mixing (starts as identity; absorbs electrode and reference differences)
  → tokens: one per channel and 50-ms patch (17 × 16 = 272) + channel and time position embeddings
  → 4 transformer layers (attention across channels and time), read out at a learned per-subject token
  → linear head → unit vector in CLIP ViT-B/32 space
loss: symmetric InfoNCE between the EEG vector and the CLIP vector of the image (THINGS) or video (ours)
```

Both datasets use the same CLIP space, so the whole network transfers, including its output layer.

**Variants** (what is trained on our data):

| variant | trained on our data |
|---|---|
| `scratch` | everything, from random weights: the same architecture without THINGS (baseline) |
| `zero-shot` | nothing: the THINGS model as it is (our subjects: identity mixing, mean THINGS subject token) |
| `frozen` | only the per-subject channel mixing and subject tokens |
| `finetune` | everything, starting from the THINGS weights, with a 3× smaller learning rate |
| control | `zero-shot` with the channels in a fixed random order: if THINGS knowledge transfers, this must fail |

Each variant runs with two **video targets** (mean CLIP vector of the frames in a window):
- `video 0–0.7 s (causal)`: visual cortex responds ~0.1 s after a change (P1 at ~100 ms), so the 0–0.8 s EEG can only reflect the frames up to 0.7 s;
- `video 0–3 s`: the whole clip.

**Validation.** The team's 5 folds of 102 held-out training videos (`video_splits.json`; 102 = size of the test set, so top-k is comparable), 4 seeds, all six subjects trained jointly. Early stopping uses 10 % of the fitting videos, so the held-out fold is never seen. Every normalisation is fitted on the fitting recordings only. **No test data are read in this notebook**; the test set is used once, in `03_evaluate_video_transformer`.

**Saved** to `~/eeg-results/pretrained-transformers/abgabe/video_transformer/<run>/`:
- `config.json`, `pretraining.csv`, `validation/` (per seed, seed ensemble, summary, similarity matrices), `figures/`;
- `models/`: a final model for **every** variant (4 seeds trained on all 1,000 training videos, plus the EEG normalisation and the config), and `chosen.json`: the variant with the best validation top-5.

The pretraining checkpoints go to `video_transformer/pretraining/<settings hash>/` and are reused by later runs with the same settings.

**Needs:** `pretrained_transformers/04_prepare_things_eeg2.ipynb` (THINGS-EEG2 download, CLIP ViT-B/32 embeddings of the THINGS images and of our video frames).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
import hashlib, json, math, os, queue, re, sys, threading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import decimate
from tqdm import tqdm                # text bars: widget bars (tqdm.auto) freeze at 0 % through the cluster's kernel gateway

## 1. Configuration

In [ ]:
# --- Paths (GPU server) ------------------------------------------------------------------------------------------------
DATA_ROOT = Path(os.environ.get('EEG_DATA_ROOT', str(Path.home() / 'eeg-data' / 'eeg-mind-reading-challenge'))).expanduser()
if (DATA_ROOT / 'eeg-mind-reading-challenge').is_dir():
    DATA_ROOT /= 'eeg-mind-reading-challenge'
THINGS_ROOT = Path(os.environ.get('THINGS_EEG2_ROOT', str(Path.home() / 'eeg-data' / 'things-eeg2'))).expanduser()
VIDEO_CLIP_DIR = Path(os.environ.get('EEG_EMBEDDING_ROOT', str(Path.home() / 'eeg-data' / 'eeg-video-embeddings'))
                      ).expanduser() / 'clip_vit_b32'          # CLIP ViT-B/32 per frame of our videos (from 04)
SPLIT_FILE = Path.cwd().resolve() / 'video_splits.json'      # team-wide folds (the GPU kernel's cwd is the cluster home)
RESULTS_ROOT = Path(os.environ.get('EEG_RESULTS_ROOT', str(Path.home() / 'eeg-results'))).expanduser()
OUT_ROOT = RESULTS_ROOT / 'pretrained-transformers' / 'video_transformer'
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUT_DIR = OUT_ROOT / RUN_ID

# --- Dataset facts -------------------------------------------------------------------------------------------------------
SUBJECTS = [1, 2, 3, 4, 5, 6]        # trained jointly: the training EEG of all six subjects may be used
TRAIN_VIDEO_IDS = np.arange(1, 1001)
N_CANDIDATES = 102                   # test candidates per subject; every validation fold has the same size
N_FOLDS, SPLIT_SEED = 5, 0           # team recipe (spectral_model/01), only used if the split file does not exist yet
EEG_SFREQ = 500
CHANNELS = ['Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2']
N_CHANNELS = len(CHANNELS)           # organisers (2026-10-18): the same channels and order for all subjects
THINGS_SUBJECTS = list(range(1, 11))
CLIP_DIM = 512                       # CLIP ViT-B/32 embedding size

# --- Input ---------------------------------------------------------------------------------------------------------------
MODEL_SFREQ = 100                    # rate of the THINGS-EEG2 release; ours is decimated 500 -> 100 Hz (team standard)
WINDOW_S = 0.8                       # THINGS epochs end at 0.8 s: the longest window both datasets have
EEG_LAG_S = 0.1                      # visual latency: cortical responses start 60-100 ms after a change (P1 ~100 ms)
CAUSAL_END_S = round(WINDOW_S - EEG_LAG_S, 6)    # the 0-0.8 s EEG can only reflect the frames up to 0.7 s
VIDEO_WINDOWS = {                    # CLIP target = mean over the frames whose display centre lies in the window
    f'video 0-{CAUSAL_END_S:g} s (causal)': (0.0, CAUSAL_END_S),
    'video 0-3 s': (0.0, 3.0),       # the whole clip
}

# --- Model ---------------------------------------------------------------------------------------------------------------
PATCH_S = 0.05                       # 50-ms tokens separate the early visual components (P1 ~0.1 s, N1 ~0.17 s)
D_MODEL, DEPTH, HEADS = 128, 4, 4    # ~0.9 M parameters: small enough for ~4,500 fine-tuning recordings, large enough
                                     # for 660k THINGS trials; head size 32 (standard)
DROPOUT = 0.1                        # standard transformer dropout
TEMPERATURE_INIT = 0.07              # CLIP's initial softmax temperature (learned afterwards)

# --- Pretraining on THINGS-EEG2 --------------------------------------------------------------------------------------------
SEEDS = [0, 1, 2, 3]                 # averaged, never selected; 4 seeds show the spread from training noise
LR = 3e-4                            # AdamW, the usual range for small transformers (1e-4 to 1e-3)
WEIGHT_DECAY = 0.05                  # standard ViT/DeiT recipe
BATCH = 1024                         # many in-batch negatives; ~650 steps per epoch on 661,600 trials
MAX_EPOCHS, PATIENCE = 60, 6         # stop after 10 % of the epoch budget without improvement
WARMUP_EPOCHS = 1                    # linear warm-up avoids large first updates of the attention weights

# --- Fine-tuning on our data -------------------------------------------------------------------------------------------------
VARIANTS = ['scratch', 'zero-shot', 'frozen', 'finetune']
CONTROL = 'shuffled (control)'       # zero-shot with the channels in a fixed random order
SETTINGS = ([(v, 'recorded', w) for v in VARIANTS for w in VIDEO_WINDOWS]        # recorded order = THINGS order
            + [('zero-shot', CONTROL, w) for w in VIDEO_WINDOWS])
FT_LR = LR / 3                       # smaller steps keep the pretrained features (common fine-tuning practice)
FT_BATCH = 256                       # ~18 steps per epoch on ~4,500 recordings; technical
FT_MAX_EPOCHS, FT_PATIENCE = 200, 20 # small data: many short epochs; patience again 10 % of the budget
INNER_FRACTION, INNER_SEED = 0.1, 0  # early stopping on 10 % of the fitting videos, the same for all variants (paired)
SELECTION_METRIC = 'top5'            # chooses the final variant: 5x more hits than top-1, so less noisy

# --- Compute -------------------------------------------------------------------------------------------------------------------
N_GPUS = torch.cuda.device_count()
FT_JOBS_PER_GPU = 3                  # technical: one small fine-tuning job does not fill a GPU
assert N_GPUS > 0, 'run this notebook on the GPU kernel'

PATCH = round(PATCH_S * MODEL_SFREQ)                     # 5 samples per token
N_SAMPLES = round(WINDOW_S * MODEL_SFREQ)                # 80 samples
THINGS_SLOTS = list(range(len(THINGS_SUBJECTS)))         # subject slots 0-9: THINGS participants, 10-15: ours
CHALLENGE_SLOT = {sub: len(THINGS_SUBJECTS) + i for i, sub in enumerate(SUBJECTS)}
N_SLOTS = len(THINGS_SUBJECTS) + len(SUBJECTS)
ARCHITECTURE = dict(n_channels=N_CHANNELS, n_samples=N_SAMPLES, patch=PATCH, d_model=D_MODEL, depth=DEPTH, heads=HEADS,
                    dropout=DROPOUT, clip_dim=CLIP_DIM, n_slots=N_SLOTS, challenge_slot=CHALLENGE_SLOT,
                    model_sfreq=MODEL_SFREQ, eeg_sfreq=EEG_SFREQ)


def tag(cfg):
    """Short hash of a settings dict: pretraining checkpoints are reused only with identical settings."""
    return hashlib.sha1(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:10]


PRETRAIN_CFG = dict(window_s=WINDOW_S, patch_s=PATCH_S, d=D_MODEL, depth=DEPTH, heads=HEADS, dropout=DROPOUT,
                    lr=LR, wd=WEIGHT_DECAY, batch=BATCH, epochs=MAX_EPOCHS, patience=PATIENCE, warmup=WARMUP_EPOCHS)
FT_CFG = dict(ft_lr=FT_LR, ft_batch=FT_BATCH, ft_epochs=FT_MAX_EPOCHS, ft_patience=FT_PATIENCE, inner=INNER_FRACTION,
              inner_seed=INNER_SEED, video_windows=VIDEO_WINDOWS, candidate_centring=True)
PRETRAIN_DIR = OUT_ROOT / 'pretraining' / tag(PRETRAIN_CFG)
for d in [PRETRAIN_DIR, OUT_DIR / 'validation' / 'similarity', OUT_DIR / 'models', OUT_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'config.json').write_text(json.dumps(dict(
    run=RUN_ID, notebook='pretrained_transformers/abgabe/01_train_video_transformer.ipynb', architecture=ARCHITECTURE,
    channels=CHANNELS, seeds=SEEDS, pretraining=PRETRAIN_CFG, finetuning=FT_CFG, settings=SETTINGS,
    versions=dict(python=sys.version.split()[0], torch=torch.__version__, numpy=np.__version__)), indent=1))
print('results ->', OUT_DIR, '\npretraining checkpoints ->', PRETRAIN_DIR)

### Helpers: validation folds, metrics, parallel GPU jobs

`run_parallel` runs independent jobs (seeds, folds, variants) on all GPUs at once and reports into **one** text progress bar with ETA. Stopping the cell stops all jobs at their next training step.

In [ ]:
if SPLIT_FILE.is_file():
    FOLDS = [np.array(f) for f in json.loads(SPLIT_FILE.read_text())['validation_folds']]
else:                                # same recipe as the other team notebooks
    perm = np.random.default_rng(SPLIT_SEED).permutation(TRAIN_VIDEO_IDS)
    FOLDS = [np.sort(perm[i * N_CANDIDATES:(i + 1) * N_CANDIDATES]) for i in range(N_FOLDS)]
    SPLIT_FILE.write_text(json.dumps(dict(seed=SPLIT_SEED, note='validation videos per fold; the other training videos '
                                          'of a fold are used for fitting', validation_folds=[f.tolist() for f in FOLDS]),
                                     indent=1))
assert len(FOLDS) == N_FOLDS and all(len(f) == N_CANDIDATES for f in FOLDS)


def metrics(S, true_cols):
    """S: similarities (recordings, candidates). The rank of the true video is its place in the ranked list a
    submission would contain (stable sort), so ties are never broken in the model's favour."""
    order = np.argsort(-np.asarray(S), axis=1, kind='stable')
    rank = np.argmax(order == np.asarray(true_cols)[:, None], axis=1) + 1
    return dict(n=len(rank), top1=float(np.mean(rank == 1)), top5=float(np.mean(rank <= 5)), mean_rank=float(rank.mean()))


def slug(setting):
    return re.sub('[^a-z0-9]+', '-', ' '.join(setting).lower()).strip('-')


STOP = threading.Event()             # set on interrupt or error: running trainings end at their next step


class Progress:
    """One shared text bar for parallel jobs (thread-safe): expect(n) adds n units to the total, tick(n) marks n units
    done, note(key, text) shows a job's status (epoch, scores) after the bar."""

    def __init__(self, bar):
        self.bar, self.lock, self.notes = bar, threading.Lock(), {}

    def expect(self, n):
        with self.lock:
            self.bar.total = (self.bar.total or 0) + n
            self.bar.refresh()

    def tick(self, n=1):
        with self.lock:
            self.bar.update(n)

    def note(self, key, text):
        with self.lock:
            self.notes[key] = text
            self.bar.set_postfix_str(' | '.join(f'{k}: {v}' for k, v in sorted(self.notes.items())), refresh=False)


def run_parallel(fn, items, jobs_per_gpu, desc, unit='run', count_items=True):
    """fn(item, progress, device) for all items, jobs_per_gpu at a time on every GPU; results in input order.
    count_items=True: one bar unit per finished item; False: the jobs report their own units (training steps)."""
    STOP.clear()
    for g in range(N_GPUS):          # CUDA's linear-algebra library loads lazily and not thread-safely: load it here
        torch.linalg.eigh(torch.eye(2, dtype=torch.float64, device=f'cuda:{g}'))
    devices = queue.Queue()
    for _ in range(jobs_per_gpu):
        for g in range(N_GPUS):
            devices.put(f'cuda:{g}')
    bar = tqdm(total=len(items) if count_items else 0, desc=desc, unit=unit)
    progress = Progress(bar)

    def on_free_gpu(item):
        device = devices.get()
        try:
            return fn(item, progress, device)
        finally:
            devices.put(device)

    out = [None] * len(items)
    pool = ThreadPoolExecutor(N_GPUS * jobs_per_gpu)
    try:                             # shuffled start order: slow and fast jobs mix, so the ETA stays steady
        futures = {pool.submit(on_free_gpu, items[i]): i for i in np.random.default_rng(0).permutation(len(items))}
        for fut in as_completed(futures):
            out[futures[fut]] = fut.result()
            if count_items:
                progress.tick()
    except BaseException:
        STOP.set()
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    finally:
        bar.close()
    pool.shutdown()
    return out

## 2. Model, loss and training loop

- **Tokens.** Each channel's 0.8 s are cut into 16 patches of 50 ms; one shared linear layer turns a patch into a token. Channel and time embeddings tell the transformer where a token comes from, so attention can relate, e.g., O1 at 100 ms to PO8 at 170 ms.
- **Loss.** In a batch, each recording should be most similar to the CLIP vector of its own stimulus, and each stimulus to its own recordings (both directions). Several recordings of the same stimulus (repetitions, subjects) all count as correct.
- **Training.** AdamW, linear warm-up, cosine decay, bfloat16 autocast (technical: faster, no effect beyond rounding). After every epoch a validation score is computed; the weights of the best epoch are kept.
- **Candidate centring** (from CORTIVA, Wang & Chen, 2026, arXiv:2608.01355, as used in the team's CORTIVA pipeline): when ranking, the mean CLIP vector of the candidates is subtracted, so what all candidates share cannot decide the ranking. It uses only the candidate videos, no EEG and no labels.

In [ ]:
class EEGTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.subject_mix = nn.Parameter(torch.eye(N_CHANNELS).repeat(N_SLOTS, 1, 1))     # per-subject channel mixing
        self.subject_token = nn.Parameter(torch.zeros(N_SLOTS, D_MODEL))                 # per-subject summary token
        self.patch_embed = nn.Linear(PATCH, D_MODEL)
        self.channel_pos = nn.Parameter(0.02 * torch.randn(N_CHANNELS, 1, D_MODEL))
        self.time_pos = nn.Parameter(0.02 * torch.randn(1, N_SAMPLES // PATCH, D_MODEL))
        layer = nn.TransformerEncoderLayer(D_MODEL, HEADS, 4 * D_MODEL, DROPOUT, activation='gelu',
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, DEPTH, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, CLIP_DIM)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / TEMPERATURE_INIT)))

    def forward(self, x, slot):                                                     # x: (batch, 17, 80)
        x = torch.bmm(self.subject_mix[slot], x)
        b, c, t = x.shape
        tokens = self.patch_embed(x.reshape(b, c, t // PATCH, PATCH))               # (batch, 17, 16, d)
        tokens = (tokens + self.channel_pos + self.time_pos).flatten(1, 2)          # (batch, 272, d)
        h = self.norm(self.encoder(torch.cat([self.subject_token[slot][:, None], tokens], dim=1))[:, 0])
        return F.normalize(self.head(h).float(), dim=-1)


INIT_LOCK = threading.Lock()         # torch.manual_seed is global: create models one at a time (reproducible weights)


def new_model(seed, device):
    with INIT_LOCK:
        torch.manual_seed(seed)
        model = EEGTransformer()
    return model.to(device)


print(f'parameters: {sum(p.numel() for p in EEGTransformer().parameters()) / 1e6:.2f} M')


def contrastive_loss(z, targets, y, logit_scale):
    logits = logit_scale.exp().clamp(max=100) * z @ targets[y].T                    # CLIP clamps the scale at 100
    pos = (y[:, None] == y[None, :]).float()                                        # same stimulus = positive
    l_eeg = -(logits.log_softmax(1) * pos).sum(1) / pos.sum(1)
    l_stim = -(logits.log_softmax(0) * pos).sum(0) / pos.sum(0)
    return (l_eeg.mean() + l_stim.mean()) / 2


@torch.no_grad()
def embed(model, X, slot, batch=4096):
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            out.append(model(X[i:i + batch], slot[i:i + batch]))
    return torch.cat(out)


def centred(cand):
    """Candidate centring: subtract the mean target over the candidates, back to unit length."""
    return F.normalize(cand - cand.mean(dim=0, keepdim=True), dim=-1)


def ranks(Z, cand, true_cols):
    """Rank of the true candidate per row (GPU; used for early stopping only)."""
    S = Z @ cand.T
    return (S > S[torch.arange(len(S), device=S.device), true_cols][:, None]).sum(1) + 1


def fit(model, X, slot, y, targets, params, lr, max_epochs, patience, batch, score_fn, seed, progress=None, name=None):
    """Contrastive training; keeps the weights of the epoch with the highest score_fn(model). With progress: reports
    every step to the shared bar (steps skipped by early stopping are counted at the end)."""
    gen = torch.Generator().manual_seed(seed)
    steps = math.ceil(len(X) / batch)
    total, warm = max_epochs * steps, WARMUP_EPOCHS * steps
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY, fused=True)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda k: min(1.0, (k + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(k, total) / total)))
    best, best_state, best_epoch, wait, history, done = -math.inf, None, 0, 0, [], 0
    if progress is not None:
        progress.expect(total)
    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(X), generator=gen).to(X.device)
        losses = []
        for i in range(0, len(X), batch):
            if STOP.is_set():
                raise RuntimeError('stopped')
            idx = order[i:i + batch]
            with torch.autocast('cuda', dtype=torch.bfloat16):
                z = model(X[idx], slot[idx])
            loss = contrastive_loss(z, targets, y[idx], model.logit_scale)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.detach())
            done += 1
            if progress is not None:
                progress.tick()
        score = score_fn(model)
        history.append(dict(epoch=epoch, loss=float(torch.stack(losses).mean()), score=score))
        if progress is not None:
            progress.note(name, f'ep {epoch} val {score:.4f} best {max(best, score):.4f}')
        if score > best:
            best, best_epoch, wait = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    if progress is not None:
        progress.tick(total - done)
    model.load_state_dict(best_state)
    return dict(best_score=best, best_epoch=best_epoch, history=history)

## 3. Pretraining on THINGS-EEG2 (seeds in parallel on the GPUs)

THINGS-EEG2 preprocessed release (prepared by `04_prepare_things_eeg2`): 10 participants, 16,540 training images × 4 repetitions and 200 test images × 80 repetitions; 17 channels, 100 Hz, 0–0.8 s. Each participant's EEG is z-scored per channel with the statistics of their training trials. Pretraining uses **single trials**, because our task is single-trial.

**Validation** = THINGS-EEG2's 200 test images (concepts not in its training set; not challenge data), single trials ranked among the 200 images (top-5 chance 2.5 %). We also report the literature-style number on EEG averaged over the 80 repetitions. A seed whose checkpoint exists is loaded, not retrained.

In [ ]:
THINGS_INFO = json.loads((THINGS_ROOT / 'compact' / 'info.json').read_text())
assert THINGS_INFO['ch_names'] == CHANNELS, 'our channel order (organisers) differs from the THINGS-EEG2 order'
assert len(THINGS_INFO['times_s']) == N_SAMPLES and round(THINGS_INFO['sfreq']) == MODEL_SFREQ

THINGS = {k: [] for k in ['tr_x', 'tr_slot', 'tr_y', 'va_x', 'va_slot', 'va_y']}
for i, s in enumerate(tqdm(THINGS_SUBJECTS, desc='Loading THINGS-EEG2', unit='participant')):
    tr = torch.from_numpy(np.load(THINGS_ROOT / 'compact' / f'sub-{s:02d}_train.npy')).to('cuda:0')   # (16540, 4, 17, 80)
    te = torch.from_numpy(np.load(THINGS_ROOT / 'compact' / f'sub-{s:02d}_test.npy')).to('cuda:0')    # (200, 80, 17, 80)
    mu = tr.double().mean(dim=(0, 1, 3), keepdim=True)
    sd = tr.double().std(dim=(0, 1, 3), keepdim=True, correction=0)
    for part, x in [('tr', tr), ('va', te)]:
        THINGS[f'{part}_x'].append(((x - mu) / sd).float().reshape(-1, N_CHANNELS, N_SAMPLES).cpu().numpy())
        THINGS[f'{part}_y'].append(np.repeat(np.arange(x.shape[0]), x.shape[1]))       # image index of each trial
        THINGS[f'{part}_slot'].append(np.full(x.shape[0] * x.shape[1], i))
THINGS = {k: np.concatenate(v) for k, v in THINGS.items()}
N_TEST_REPS = te.shape[1]


def unit(a):
    return a / np.linalg.norm(a, axis=-1, keepdims=True)


THINGS_TARGETS = {'tr': unit(np.load(THINGS_ROOT / 'clip_vit_b32' / 'things_train.npy')),
                  'va': unit(np.load(THINGS_ROOT / 'clip_vit_b32' / 'things_test.npy'))}
print(f"THINGS: {len(THINGS['tr_x'])} training trials, {len(THINGS['va_x'])} validation trials "
      f"({len(THINGS_TARGETS['va'])} held-out images)")

In [ ]:
def pretrain(seed, progress, device):
    ckpt, meta_path = PRETRAIN_DIR / f'seed{seed}.pt', PRETRAIN_DIR / f'seed{seed}.json'
    if ckpt.is_file() and meta_path.is_file():
        return json.loads(meta_path.read_text())
    t = lambda a, dtype=torch.float32: torch.as_tensor(a, dtype=dtype, device=device)
    X, slot, y = t(THINGS['tr_x']), t(THINGS['tr_slot'], torch.long), t(THINGS['tr_y'], torch.long)
    VX, vslot, vy = t(THINGS['va_x']), t(THINGS['va_slot'], torch.long), t(THINGS['va_y'], torch.long)
    targets, vtargets = t(THINGS_TARGETS['tr']), t(THINGS_TARGETS['va'])

    def score(model):                # single-trial top-5 among the 200 held-out images
        return float((ranks(embed(model, VX, vslot), vtargets, vy) <= 5).float().mean())

    model = new_model(seed, device)
    result = fit(model, X, slot, y, targets, model.parameters(), LR, MAX_EPOCHS, PATIENCE, BATCH, score, seed,
                 progress, f'seed {seed}')
    r_single = ranks(embed(model, VX, vslot), vtargets, vy)
    n = len(THINGS_SUBJECTS)         # literature-style: average the 80 repetitions per image and participant first
    first = lambda a: a.reshape(n, -1, N_TEST_REPS)[:, :, 0].flatten()
    VX_avg = VX.reshape(n, -1, N_TEST_REPS, N_CHANNELS, N_SAMPLES).mean(2).flatten(0, 1)
    r_avg = ranks(embed(model, VX_avg, first(vslot)), vtargets, first(vy))
    result |= dict(seed=seed, single_top1=float((r_single == 1).float().mean()),
                   single_top5=float((r_single <= 5).float().mean()),
                   averaged_top1=float((r_avg == 1).float().mean()), averaged_top5=float((r_avg <= 5).float().mean()))
    torch.save(model.state_dict(), ckpt)
    meta_path.write_text(json.dumps(result, indent=1))
    return result


# one pretraining per GPU: each holds its own copy of the THINGS trials (~5 GB)
PRETRAIN = run_parallel(pretrain, SEEDS, 1, 'Pretraining (all seeds)', 'step', count_items=False)
PRE_TABLE = pd.DataFrame([{k: v for k, v in r.items() if k != 'history'} for r in PRETRAIN])
PRE_TABLE.to_csv(OUT_DIR / 'pretraining.csv', index=False)
print(PRE_TABLE.round(4).to_string(index=False))
print('chance (200 images): top-1 0.005, top-5 0.025')

fig, ax = plt.subplots(figsize=(6, 3.5))
for r in PRETRAIN:
    h = pd.DataFrame(r['history'])
    ax.plot(h.epoch, h.score, label=f"seed {r['seed']}")
ax.axhline(5 / len(THINGS_TARGETS['va']), color='0.4', ls='--', lw=1, label='chance')
ax.set_xlabel('epoch')
ax.set_ylabel('THINGS validation top-5 (single trial)')
ax.set_title('Pretraining on THINGS-EEG2', fontsize=10)
ax.legend(fontsize=8)
fig.savefig(OUT_DIR / 'figures' / 'pretraining_curves.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Our data in the THINGS format

Per subject, as close as possible to the THINGS preprocessing:
1. remove each channel's offset per recording; decimate 500 → 100 Hz (FIR, zero phase, as in the other team notebooks);
2. **whitening** with the symmetric inverse square root of the channel covariance, with Ledoit-Wolf shrinkage as in the THINGS preprocessing. The symmetric form changes each channel as little as possible, so channel identities survive;
3. keep 0–0.8 s and z-score each channel.

Steps 2–3 are fitted on the **fitting recordings of the current fold only** and then applied to the held-out ones. The final models store these statistics, so `03` applies the training statistics to the test EEG.

**Video targets:** CLIP ViT-B/32 of 32 frames per video (from `04`), averaged over the frames of the window, unit length.

In [ ]:
def load_training_eeg(sub):
    """Video IDs and training recordings (n, 17, 300) of one subject at 100 Hz, float64."""
    with np.load(DATA_ROOT / 'trn' / f'sub-{sub:02d}_trn.npz') as z:
        keys = sorted(z.files, key=int)
        ids, eeg = np.array([int(k) for k in keys]), np.stack([z[k] for k in keys]).astype(np.float64)
    eeg -= eeg.mean(axis=2, keepdims=True)
    return ids, decimate(eeg, EEG_SFREQ // MODEL_SFREQ, axis=-1, ftype='fir', zero_phase=True)


RAW = {sub: load_training_eeg(sub) for sub in tqdm(SUBJECTS, desc='Loading our training EEG', unit='subject')}
ORDERS = {'recorded': np.arange(N_CHANNELS),                                     # = THINGS order (asserted above)
          CONTROL: np.random.default_rng(0).permutation(N_CHANNELS)}             # fixed random order (control)
print('control channel order:', [CHANNELS[i] for i in ORDERS[CONTROL]])


def ledoit_wolf(X):
    """Shrunk covariance of samples X (n, p), float64 torch: the formula of sklearn.covariance.LedoitWolf."""
    X = X - X.mean(0)
    n, p = X.shape
    emp = X.T @ X / n
    mu = torch.trace(emp) / p
    X2 = X ** 2
    delta_ = ((X.T @ X) ** 2).sum() / n ** 2
    beta = ((X2.T @ X2).sum() / n - delta_) / (p * n)
    delta = (delta_ - 2 * mu * torch.trace(emp) + p * mu ** 2) / p
    shrinkage = 0.0 if beta == 0 else float(torch.minimum(beta, delta) / delta)
    return (1 - shrinkage) * emp + shrinkage * mu * torch.eye(p, dtype=X.dtype, device=X.device)


def fit_normaliser(eeg_fit):
    """Whitening matrix W and per-channel mean/std after whitening, from (n, 17, samples) float64 GPU recordings."""
    w, V = torch.linalg.eigh(ledoit_wolf(eeg_fit.transpose(1, 2).reshape(-1, N_CHANNELS)))
    W = V @ torch.diag(w.rsqrt()) @ V.T
    white = torch.einsum('cd,ndt->nct', W, eeg_fit)[:, :, :N_SAMPLES]
    return W, white.mean(dim=(0, 2), keepdim=True), white.std(dim=(0, 2), keepdim=True, correction=0)


def normalise(eeg, stats):
    """Recordings (n, 17, samples) -> float32 model inputs (n, 17, 80)."""
    W, mu, sd = stats
    return ((torch.einsum('cd,ndt->nct', W, eeg)[:, :, :N_SAMPLES] - mu) / sd).float()


with np.load(VIDEO_CLIP_DIR / 'embeddings.npz') as z:
    FRAME_EMB = {int(k): z[k] for k in z.files}
FRAME_T = np.array(json.loads((VIDEO_CLIP_DIR / 'metadata.json').read_text())['frame_times_s'])
ALL_VIDEO_IDS = np.array(sorted(FRAME_EMB))
ROW_OF = np.full(ALL_VIDEO_IDS.max() + 1, -1)
ROW_OF[ALL_VIDEO_IDS] = np.arange(len(ALL_VIDEO_IDS))
FRAME_CENTRE = FRAME_T + np.median(np.diff(FRAME_T)) / 2        # a frame counts at the centre of its display time
VIDEO_TARGETS = {}
for name, (a, b) in VIDEO_WINDOWS.items():
    keep = (FRAME_CENTRE >= a) & (FRAME_CENTRE < b)
    VIDEO_TARGETS[name] = unit(np.stack([FRAME_EMB[v][keep].mean(0) for v in ALL_VIDEO_IDS]))
    print(f'{name}: {keep.sum()} frames per video')

## 5. Validation of all variants (5 folds × 4 seeds)

One job = setting × fold × seed. Seed *s* of a pretrained variant starts from pretraining seed *s*. Each recording of the held-out fold is ranked among the fold's 102 videos: the same task as on the test set.

Early stopping: 10 % of the fitting videos (the same for all variants and seeds, so comparisons are paired) form an inner set; after each epoch, the mean rank of their recordings among these videos decides.

In [ ]:
def subject_inputs(fit_videos, order, device, query_videos=None):
    """Model inputs of all six subjects, normalised per subject with its fitting recordings. Returns the fitting inputs
    (X, slot, video ID), the query inputs {sub: (X, video IDs)} and the normalisation statistics {sub: stats}."""
    X, slot, vid, queries, stats = [], [], [], {}, {}
    for sub in SUBJECTS:
        ids, eeg = RAW[sub]
        eeg = torch.as_tensor(eeg[:, order], device=device)
        fit_rows = np.isin(ids, fit_videos)
        fit_eeg = eeg[torch.as_tensor(fit_rows, device=device)]
        stats[sub] = fit_normaliser(fit_eeg)
        X.append(normalise(fit_eeg, stats[sub]))
        vid.append(ids[fit_rows])
        slot.append(torch.full((int(fit_rows.sum()),), CHALLENGE_SLOT[sub], device=device))
        if query_videos is not None:
            q = np.isin(ids, query_videos)
            queries[sub] = (normalise(eeg[torch.as_tensor(q, device=device)], stats[sub]), ids[q])
    return torch.cat(X), torch.cat(slot), np.concatenate(vid), queries, stats


def init_model(variant, seed, device):
    model = new_model(seed, device)
    if variant != 'scratch':
        model.load_state_dict(torch.load(PRETRAIN_DIR / f'seed{seed}.pt', map_location=device, weights_only=True))
        with torch.no_grad():        # our subjects start from identity mixing and the mean THINGS subject token
            slots = list(CHALLENGE_SLOT.values())
            model.subject_mix[slots] = torch.eye(N_CHANNELS, device=device)
            model.subject_token[slots] = model.subject_token[THINGS_SLOTS].mean(0)
    return model


def train_setting(setting, seed, fit_videos, device, query_videos=None):
    """Train one setting (variant, channel order, video window) on fit_videos minus an inner early-stopping set."""
    variant, order_name, window = setting
    X, slot, vid, queries, stats = subject_inputs(fit_videos, ORDERS[order_name], device, query_videos)
    model = init_model(variant, seed, device)
    if variant == 'zero-shot':
        return model, queries, stats, dict(best_epoch=0)
    targets = torch.as_tensor(VIDEO_TARGETS[window], device=device)
    inner = np.sort(np.random.default_rng(INNER_SEED).choice(fit_videos, round(INNER_FRACTION * len(fit_videos)),
                                                             replace=False))
    is_inner = np.isin(vid, inner)
    m = torch.as_tensor(is_inner, device=device)
    inner_cand = centred(targets[torch.as_tensor(ROW_OF[inner], device=device)])
    inner_true = torch.as_tensor(np.searchsorted(inner, vid[is_inner]), device=device)

    def score(model):                # higher = better: minus the mean rank of the inner recordings among the inner videos
        return -float(ranks(embed(model, X[m], slot[m]), inner_cand, inner_true).float().mean())

    params = [model.subject_mix, model.subject_token] if variant == 'frozen' else model.parameters()
    y = torch.as_tensor(ROW_OF[vid], device=device)
    info = fit(model, X[~m], slot[~m], y[~m], targets, params, FT_LR if variant == 'finetune' else LR,
               FT_MAX_EPOCHS, FT_PATIENCE, FT_BATCH, score, seed)
    return model, queries, stats, info


def similarity(model, x, sub, cand):
    slot = torch.full((len(x),), CHALLENGE_SLOT[sub], device=x.device)
    return (embed(model, x, slot) @ cand.T).cpu().numpy()


def validation_job(job, progress, device):
    setting, f, seed = job
    fold = FOLDS[f]
    model, queries, _, info = train_setting(setting, seed, np.setdiff1d(TRAIN_VIDEO_IDS, fold), device, fold)
    cand = centred(torch.as_tensor(VIDEO_TARGETS[setting[2]][ROW_OF[fold]], device=device))
    return dict(setting=setting, fold=f, seed=seed, best_epoch=info['best_epoch'],
                sims={sub: similarity(model, x, sub, cand) for sub, (x, _) in queries.items()},
                truth={sub: np.searchsorted(fold, ids) for sub, (_, ids) in queries.items()})


jobs = [(s, f, seed) for s in SETTINGS for f in range(N_FOLDS) for seed in SEEDS]
RUNS = run_parallel(validation_job, jobs, FT_JOBS_PER_GPU, 'Validation runs')

### Validation results

Two numbers per setting, both the mean over subjects and folds: **per seed** (one trained model) and **seed ensemble** (similarities averaged over the 4 seeds, as in the final model). Chance: top-1 1/102 = 0.010, top-5 5/102 = 0.049.

In [ ]:
NAMES = ['variant', 'channel_order', 'video_window']
per_seed, grouped = [], {}
for r in RUNS:
    for sub in SUBJECTS:
        S, truth = r['sims'][sub], r['truth'][sub]
        per_seed.append(dict(zip(NAMES, r['setting']), fold=r['fold'], seed=r['seed'], subject=sub,
                             best_epoch=r['best_epoch'], **metrics(S, truth)))
        grouped.setdefault((r['setting'], r['fold'], sub), []).append((S, truth))
ENSEMBLE_SIMS = {key: (np.mean([S for S, _ in v], axis=0), v[0][1]) for key, v in grouped.items()}
PER_SEED = pd.DataFrame(per_seed)
ENSEMBLE = pd.DataFrame([dict(zip(NAMES, s), fold=f, subject=sub, **metrics(S, truth))
                         for (s, f, sub), (S, truth) in ENSEMBLE_SIMS.items()])
SUMMARY = (ENSEMBLE.groupby(NAMES)[['top1', 'top5', 'mean_rank']].mean()
           .join(PER_SEED.groupby(NAMES)[['top5', 'best_epoch']].mean().rename(columns={'top5': 'top5_per_seed'})))
for name, table in [('per_seed', PER_SEED), ('seed_ensemble', ENSEMBLE), ('summary', SUMMARY.reset_index())]:
    table.to_csv(OUT_DIR / 'validation' / f'{name}.csv', index=False)
for setting in SETTINGS:             # seed-ensemble similarities, e.g. to fit a fusion with other models on validation
    np.savez(OUT_DIR / 'validation' / 'similarity' / f'{slug(setting)}.npz',
             **{f'fold{f}_sub{sub:02d}{part}': a for (s, f, sub), pair in ENSEMBLE_SIMS.items() if s == setting
                for part, a in zip(['', '_truth'], pair)})
SUMMARY.round(3)

In [ ]:
labels = [' | '.join(k) for k in SUMMARY.index]
per_sub = ENSEMBLE.groupby(NAMES + ['subject'])['top5'].mean().unstack('subject').loc[SUMMARY.index]
colors = {v: f'C{i}' for i, v in enumerate(VARIANTS)}
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(range(len(labels)), SUMMARY.top5, color=[colors[k[0]] for k in SUMMARY.index], alpha=0.7)
for i, vals in enumerate(per_sub.values):
    ax.plot(np.full(len(vals), i), vals, 'k.', ms=3)
ax.axhline(5 / N_CANDIDATES, color='0.4', ls='--', lw=1, label='chance')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)
ax.set_ylabel('validation top-5 (seed ensemble)')
ax.set_title('Does THINGS-EEG2 pretraining help? (colour = variant, dots = subjects)', fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / 'figures' / 'validation_variants.png', dpi=130, bbox_inches='tight')
plt.show()

**How to read it.** `finetune` above `scratch`: THINGS pretraining transfers. `frozen` close to `finetune`: only the channel mapping must be learned. `zero-shot` above chance with the recorded order but not with the shuffled control: the THINGS model transfers and needs the correct channels. Causal window above `0–3 s`: the early EEG reflects what was on screen just before.

**Reference result** (same pipeline in `pretrained_transformers/05`, run 2026-10-18; a rerun differs slightly, GPU training is not bit-exact), causal window, seed ensemble:

| variant | top-1 | top-5 |
|---|---|---|
| `finetune` | 0.064 | 0.229 |
| `frozen` | 0.023 | 0.111 |
| `scratch` | 0.017 | 0.078 |
| `zero-shot` | 0.013 | 0.064 |
| `zero-shot`, shuffled channels | 0.013 | 0.059 |

- **Pretraining is what makes the transformer work:** `finetune` reaches ~3× the top-5 of `scratch`.
- `frozen` reaches about half of `finetune`: the THINGS features help as they are, but going from static object images (100 ms each) to natural videos needs the whole network to adapt. Its best epoch was ~160 of 200, so it was still improving: its score is a lower bound.
- `zero-shot` is only slightly above chance and above the shuffled control: without adaptation, the model hardly transfers (other amplifier, reference, stimulus type).
- The causal 0–0.7 s target and the whole clip score about the same (0.229 vs 0.223 top-5): the CLIP content of a 3-s clip changes little over time, so its first frames already describe it.
- Pretraining itself: single-trial top-5 ~0.25 among the 200 THINGS test images (chance 0.025), ~0.42–0.50 on averaged trials.

## 6. Final models: every setting trained on all 1,000 training videos

Each setting is trained again on all training videos (10 % inner set for early stopping, as in validation), once per seed. Saved per setting:
- `models/<setting>.pt`: the 4 state dicts and each subject's EEG normalisation (whitening, mean, std; training recordings only);
- `models/<setting>.json`: setting, channel order, video window, best epochs and validation scores.

`models/chosen.json` names the setting with the best validation top-5 (never the control); `03` uses it for the submission. The test set is not read here.

In [ ]:
def final_job(job, progress, device):
    setting, seed = job
    model, _, stats, info = train_setting(setting, seed, TRAIN_VIDEO_IDS, device)
    return dict(state={k: v.detach().cpu() for k, v in model.state_dict().items()}, best_epoch=info['best_epoch'],
                normalisation={sub: [t.cpu() for t in s] for sub, s in stats.items()})


final_jobs = [(s, seed) for s in SETTINGS for seed in SEEDS]
FINAL = dict(zip(final_jobs, run_parallel(final_job, final_jobs, FT_JOBS_PER_GPU, 'Final models')))
for setting in SETTINGS:
    runs = [FINAL[(setting, seed)] for seed in SEEDS]
    torch.save(dict(states={seed: r['state'] for seed, r in zip(SEEDS, runs)},
                    normalisation=runs[0]['normalisation']),             # the same for all seeds (same recordings)
               OUT_DIR / 'models' / f'{slug(setting)}.pt')
    (OUT_DIR / 'models' / f'{slug(setting)}.json').write_text(json.dumps(dict(
        name=slug(setting), **dict(zip(NAMES, setting)), channel_index=ORDERS[setting[1]].tolist(),
        video_window_s=VIDEO_WINDOWS[setting[2]], seeds=SEEDS, best_epochs=[r['best_epoch'] for r in runs],
        validation=SUMMARY.loc[setting, ['top1', 'top5', 'mean_rank']].round(4).to_dict()), indent=1))

best = SUMMARY.drop(index=CONTROL, level='channel_order')[SELECTION_METRIC].idxmax()
(OUT_DIR / 'models' / 'chosen.json').write_text(json.dumps(dict(
    name=slug(best), **dict(zip(NAMES, best)), rule=f'best validation {SELECTION_METRIC} (seed ensemble), control excluded',
    validation=SUMMARY.loc[best, ['top1', 'top5', 'mean_rank']].round(4).to_dict()), indent=1))
print(f'{len(SETTINGS)} final models saved in {OUT_DIR / "models"}\nchosen on validation: {slug(best)}')